# CropGuide AI — JSON Validation Pipeline
## Live Instructor Exercise: Read → Validate → Log → Write

You will build a small data-engineering pipeline that:

1. reads a JSON file containing farmer records
2. checks each record for required keys
3. logs a `WARNING` for any missing key
4. skips invalid records
5. writes only valid records to a new JSON file
6. logs a final summary with total, valid, and skipped counts

## CropGuide scenario

CropGuide receives farmer-registration records from another system.

A valid record must contain:

- `farmer_id`
- `name`
- `country`
- `crop`
- `farm_size_hectares`

Pipeline flow:

```text
raw JSON
   ↓
read records
   ↓
check required keys
   ↓
valid? ── yes ──> keep record
   │
   no
   ↓
log WARNING
   ↓
skip record
   ↓
write valid records
   ↓
log summary
```

# 1. Knowledge needed before the pipeline

You should understand:

- dictionaries
- lists of dictionaries
- loops
- functions
- `if` statements
- `in` / `not in`
- `enumerate()`
- `continue`
- file paths
- `with open(...)`
- `json.load()`
- `json.dump()`
- counters
- Python logging
- `logging.basicConfig()`
- `logging.warning()`
- `logging.info()`

# 2. One dictionary = one record

In [ ]:
farmer = {
    "farmer_id": 1001,
    "name": "Amina Yusuf",
    "country": "Nigeria",
    "crop": "Tomato",
    "farm_size_hectares": 2.5
}

farmer

In [ ]:
print(farmer["crop"])
print(farmer["country"])

# 3. A list of dictionaries = a batch of records

In [ ]:
farmers = [
    {
        "farmer_id": 1001,
        "name": "Amina Yusuf",
        "country": "Nigeria",
        "crop": "Tomato",
        "farm_size_hectares": 2.5
    },
    {
        "farmer_id": 1002,
        "name": "Kojo Mensah",
        "country": "Ghana",
        "crop": "Maize",
        "farm_size_hectares": 4.0
    }
]

farmers

# 4. Define required keys

In [ ]:
REQUIRED_KEYS = [
    "farmer_id",
    "name",
    "country",
    "crop",
    "farm_size_hectares"
]

REQUIRED_KEYS

# 5. Check whether a key exists

In [ ]:
sample_record = {
    "farmer_id": 1001,
    "name": "Amina Yusuf",
    "country": "Nigeria"
}

print("name" in sample_record)
print("crop" in sample_record)

# 6. Find missing keys

In [ ]:
sample_record = {
    "farmer_id": 1001,
    "name": "Amina Yusuf",
    "country": "Nigeria"
}

missing_keys = []

for key in REQUIRED_KEYS:
    if key not in sample_record:
        missing_keys.append(key)

print(missing_keys)

# 7. Put the missing-key logic inside a function

In [ ]:
def get_missing_keys(record, required_keys):
    missing_keys = []

    for key in required_keys:
        if key not in record:
            missing_keys.append(key)

    return missing_keys

In [ ]:
test_record = {
    "farmer_id": 1003,
    "name": "Njeri Kamau",
    "crop": "Beans"
}

print(get_missing_keys(test_record, REQUIRED_KEYS))

# 8. File I/O

Use `with open(...)` so Python closes files automatically.

Modes:

- `"r"` = read
- `"w"` = write
- `"a"` = append

For this exercise:

```text
input JSON  → read mode
output JSON → write mode
```

# 9. JSON: load and dump

In [ ]:
import json

`json.load(file)`:

```text
JSON file → Python object
```

`json.dump(data, file)`:

```text
Python object → JSON file
```

# 10. Create CropGuide practice data

In [ ]:
raw_records = [
    {
        "farmer_id": 1001,
        "name": "Amina Yusuf",
        "country": "Nigeria",
        "crop": "Tomato",
        "farm_size_hectares": 2.5
    },
    {
        "farmer_id": 1002,
        "name": "Kojo Mensah",
        "country": "Ghana",
        "crop": "Maize",
        "farm_size_hectares": 4.0
    },
    {
        "farmer_id": 1003,
        "name": "Njeri Kamau",
        "country": "Kenya",
        "farm_size_hectares": 1.8
    },
    {
        "farmer_id": 1004,
        "name": "Chinedu Okafor",
        "crop": "Cassava",
        "farm_size_hectares": 6.2
    },
    {
        "farmer_id": 1005,
        "country": "Nigeria",
        "crop": "Rice",
        "farm_size_hectares": 3.1
    }
]

raw_records

# 11. Write the practice input file

In [ ]:
from pathlib import Path

data_dir = Path("data")
data_dir.mkdir(exist_ok=True)

input_path = data_dir / "cropguide_raw_farmers.json"

with open(input_path, "w", encoding="utf-8") as file:
    json.dump(raw_records, file, indent=4)

print(f"Created: {input_path}")

# 12. Read the JSON file

In [ ]:
with open(input_path, "r", encoding="utf-8") as file:
    records = json.load(file)

print(type(records))
print("Total records:", len(records))
records

# 13. Logging basics

Common log levels:

```text
DEBUG
INFO
WARNING
ERROR
CRITICAL
```

For this pipeline:

- `INFO` = normal progress / summary
- `WARNING` = incomplete record skipped

In [ ]:
import logging

In [ ]:
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(message)s"
)

In [ ]:
logging.info("CropGuide pipeline started.")
logging.warning("Example warning: incomplete record.")

# 14. Build the validation loop manually

In [ ]:
valid_records = []
skipped_count = 0

for index, record in enumerate(records):
    missing_keys = get_missing_keys(record, REQUIRED_KEYS)

    if missing_keys:
        logging.warning(
            "Record %s skipped. Missing keys: %s",
            index,
            missing_keys
        )
        skipped_count += 1
        continue

    valid_records.append(record)

print("Valid records:", len(valid_records))
print("Skipped records:", skipped_count)

Important ideas:

- `enumerate(records)` gives both index and record
- `if missing_keys:` is true when at least one required key is missing
- `continue` skips the rest of the current loop and moves to the next record

# 15. Write only valid records

In [ ]:
output_path = data_dir / "cropguide_valid_farmers.json"

with open(output_path, "w", encoding="utf-8") as file:
    json.dump(valid_records, file, indent=4)

print(f"Created: {output_path}")

# 16. Log the summary

In [ ]:
total_count = len(records)
valid_count = len(valid_records)

logging.info(
    "Pipeline summary | total=%s | valid=%s | skipped=%s",
    total_count,
    valid_count,
    skipped_count
)

# 17. Build the complete pipeline function

In [ ]:
def process_json_pipeline(input_file, output_file, required_keys):
    # Read input JSON
    with open(input_file, "r", encoding="utf-8") as file:
        records = json.load(file)

    valid_records = []
    skipped_count = 0

    # Validate each record
    for index, record in enumerate(records):
        missing_keys = get_missing_keys(record, required_keys)

        if missing_keys:
            logging.warning(
                "Record %s skipped. Missing keys: %s",
                index,
                missing_keys
            )
            skipped_count += 1
            continue

        valid_records.append(record)

    # Write valid records only
    with open(output_file, "w", encoding="utf-8") as file:
        json.dump(valid_records, file, indent=4)

    total_count = len(records)
    valid_count = len(valid_records)

    # Final summary
    logging.info(
        "Pipeline summary | total=%s | valid=%s | skipped=%s",
        total_count,
        valid_count,
        skipped_count
    )

# 18. Run the final pipeline

In [ ]:
final_output_path = data_dir / "cropguide_valid_farmers_final.json"

process_json_pipeline(
    input_file=input_path,
    output_file=final_output_path,
    required_keys=REQUIRED_KEYS
)

# 19. Verify the output

In [ ]:
with open(final_output_path, "r", encoding="utf-8") as file:
    final_records = json.load(file)

print("Records written:", len(final_records))
final_records

Expected counts for this sample:

```text
total   = 5
valid   = 2
skipped = 3
```

# 20. Refactor validation into a clearer function

In [ ]:
def validate_required_keys(record, required_keys):
    missing_keys = [
        key
        for key in required_keys
        if key not in record
    ]

    if missing_keys:
        return False, missing_keys

    return True, []

# 21. Refactored pipeline

In [ ]:
def process_json_pipeline_v2(input_file, output_file, required_keys):
    with open(input_file, "r", encoding="utf-8") as file:
        records = json.load(file)

    valid_records = []
    skipped_count = 0

    for index, record in enumerate(records):
        is_valid, missing_keys = validate_required_keys(
            record,
            required_keys
        )

        if not is_valid:
            logging.warning(
                "Record %s skipped. Missing keys: %s",
                index,
                missing_keys
            )
            skipped_count += 1
            continue

        valid_records.append(record)

    with open(output_file, "w", encoding="utf-8") as file:
        json.dump(valid_records, file, indent=4)

    logging.info(
        "Pipeline summary | total=%s | valid=%s | skipped=%s",
        len(records),
        len(valid_records),
        skipped_count
    )

# 22. Optional next step: file logging

In a normal Python script:

```python
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(message)s",
    handlers=[
        logging.FileHandler("pipeline.log"),
        logging.StreamHandler()
    ]
)
```

This writes logs to both the terminal and a file.

# 23. Presence validation vs type validation vs business rules

This exercise checks **presence** only.

Example:

```python
{"farm_size_hectares": -10}
```

The key exists, so it passes today's required-key validation.

Later you can add:

```text
Presence validation
→ is the key there?

Type validation
→ is farm_size_hectares numeric?

Business-rule validation
→ is farm_size_hectares >= 0?
```

# 24. Record failure vs pipeline failure

A bad record can often be handled with:

```text
WARNING → skip → continue
```

But pipeline-level problems such as:

```text
file not found
broken JSON
permission denied
```

may require the whole pipeline to stop.

# 25. Optional file-level error handling

In [ ]:
def safe_read_json(input_file):
    try:
        with open(input_file, "r", encoding="utf-8") as file:
            return json.load(file)

    except FileNotFoundError:
        logging.error("Input file not found: %s", input_file)
        raise

    except json.JSONDecodeError:
        logging.error("Invalid JSON in file: %s", input_file)
        raise

# 26. How this is already ETL

```text
EXTRACT
json.load()

TRANSFORM
validate + filter

LOAD
json.dump()
```

So this simple exercise is already a small ETL pipeline.

# 27. How this connects to later data engineering

```text
dictionary validation
    ↓
Pydantic

required-key checks
    ↓
schema contracts

manual pipeline function
    ↓
Airflow tasks

logging
    ↓
Airflow logs / observability

JSON
    ↓
REST APIs

valid/rejected records
    ↓
data-quality frameworks
```

# 28. Instructor live-teaching order

Build the exercise in this exact sequence:

1. one farmer dictionary
2. list of farmer dictionaries
3. `REQUIRED_KEYS`
4. `"crop" in record`
5. manually find missing keys
6. create `get_missing_keys()`
7. create the JSON input file
8. read it with `json.load()`
9. introduce logging
10. configure `basicConfig()`
11. loop through records
12. log `WARNING`
13. use `continue`
14. append valid records
15. write valid records with `json.dump()`
16. calculate counts
17. log summary
18. wrap everything inside a function

# 29. Student independent practice

Create at least 8 farmer records:

- at least 4 valid
- one missing `country`
- one missing `crop`
- one missing `name`
- one missing `farm_size_hectares`

Then:

1. read the file
2. validate required keys
3. log warnings for rejected records
4. write only valid records
5. log total, valid, and skipped counts

Do not hard-code the counts.

# 30. Knowledge check

Be able to answer:

1. What does one dictionary represent?
2. What does a list of dictionaries represent?
3. What does `key in record` check?
4. What does `json.load()` do?
5. What does `json.dump()` do?
6. Why use `with open(...)`?
7. What is the difference between `INFO` and `WARNING`?
8. What does `continue` do?
9. Why track `skipped_count`?
10. Why write valid records to another file?
11. What does the summary log tell us?
12. What is the difference between a bad record and a pipeline failure?
13. How is this ETL?
14. What would type validation add?

# Final mental model

```text
JSON input
   ↓
json.load()
   ↓
list of dictionaries
   ↓
for loop
   ↓
required-key validation
   ↓
valid? ────────────────┐
   │                    │
  yes                  no
   │                    │
append             WARNING log
   │                    │
   │                 skipped += 1
   │                    │
   └────────────┬───────┘
                ↓
        write valid records
                ↓
           json.dump()
                ↓
        final INFO summary
```

If you understand this flow, you understand the core logic of a small validation pipeline.